# Previsão de Preços — Perguntas de Negócio (Ações e Criptomoedas)

9 perguntas de negócio respondidas com base nos 3 modelos treinados na Fase 5 (Regressão Linear, Random Forest, XGBoost) sobre 8 anos de dados de 12 tickers (2018–2026). O LSTM (Fase 5, 4/4) está pendente — as respostas actualizam-se quando estiver concluído.

**Regra do notebook:** respostas em tabelas, sem gráficos.

| # | Pergunta | Base |
|---|----------|------|
| Q1 | Qual dos modelos prevê melhor o preço de fecho de amanhã? | `preds_linreg` / `preds_rf` / `preds_xgb` |
| Q2 | Em que tickers a previsão é mais fiável? | preds (validação) |
| Q3 | O modelo acerta a direcção do mercado? | preds + `close` do dia t |
| Q4 | As previsões geram retorno superior ao buy & hold? | Backtest no teste (2024+) |
| Q5 | Funciona melhor em acções, ETFs ou criptomoedas? | preds por classe de activo |
| Q6 | O erro aumenta com o tempo? | MAPE por ano (val → test) |
| Q7 | Erra mais quando o mercado sobe ou desce? | Viés do erro por direcção |
| Q8 | Quais as features mais importantes? | Re-treino RF/XGBoost (Fase 5) |
| Q9 | Qual o pior cenário de erro por ticker? | Cauda do erro (P95) no teste |

Estrutura: STEP 1 (base única de previsões) → Q1–Q9 (uma pergunta por célula).

In [0]:
%pip install xgboost

In [0]:
# ==============================================================================
# STEP 1: DADOS — BASE ÚNICA DE PREVISÕES DOS 3 MODELOS
# ==============================================================================
# Objectivo: Consolidar as previsões da Fase 5 (LinReg, RF, XGBoost) numa base
# única, com o fecho do dia t (para direcção e backtest), o erro por sessão e a
# classe de activo — a base de todas as perguntas de negócio.
#
# Input:  workspace.market_project.preds_linreg / preds_rf / preds_xgb
#         workspace.market_project.gold_features_split (close do dia t)
# Output: DataFrame pandas `preds` (3 modelos × 15.636 sessões val+test)
#
# Operações:
#   - Union das previsões val+test dos 3 modelos com coluna `modelo`
#   - Join com o `close` do dia t (base do sinal de subida/descida)
#   - Cálculo do erro, erro %, direcção real/prevista, retorno diário e classe

import pandas as pd
import numpy as np

SCHEMA = "workspace.market_project"

TABELAS = {"linreg": "preds_linreg", "rf": "preds_rf", "xgb": "preds_xgb"}

frames = []
for modelo, tabela in TABELAS.items():
    df = spark.table(f"{SCHEMA}.{tabela}").select("ticker", "date", "split", "y_real", "y_prev").toPandas()
    df["modelo"] = modelo
    frames.append(df)
preds = pd.concat(frames, ignore_index=True)

fechos = spark.table(f"{SCHEMA}.gold_features_split").select("ticker", "date", "close").toPandas()
preds = preds.merge(fechos, on=["ticker", "date"], how="left")
assert preds["close"].isna().sum() == 0, "Fechamentos em falta!"

preds["erro"] = preds["y_prev"] - preds["y_real"]              # > 0: previu acima do real
preds["ape"] = preds["erro"].abs() / preds["y_real"] * 100       # erro % absoluto
preds["dir_real"] = np.sign(preds["y_real"] - preds["close"])   # subida/descida real t→t+1
preds["dir_prev"] = np.sign(preds["y_prev"] - preds["close"])   # sinal previsto
preds["acertou"] = preds["dir_real"] == preds["dir_prev"]
preds["ret_dia"] = preds["y_real"] / preds["close"] - 1          # retorno realizado t→t+1

CLASSES = {"BTC-USD": "Cripto", "ETH-USD": "Cripto",
           "SPY": "ETF", "QQQ": "ETF", "IAU": "ETF"}
preds["classe"] = preds["ticker"].map(CLASSES).fillna("Acções")

MODELOS_PT = {"linreg": "LinReg", "rf": "RF", "xgb": "XGBoost"}
preds["modelo_pt"] = preds["modelo"].map(MODELOS_PT)

n_sessoes = len(preds) // 3
print(f"Base carregada: {len(preds):,} linhas = 3 modelos × {n_sessoes:,} sessões (val + test)")
print(f"Tickers: {preds['ticker'].nunique()} | Classes: {sorted(preds['classe'].unique())}")
print("✅ Erro, direcção, retorno diário e classe calculados por sessão/modelo")

In [0]:
# ==============================================================================
# Q1: QUAL DOS MODELOS PREVÊ MELHOR O PREÇO DE FECHO DE AMANHÃ?
# ==============================================================================
# Descrição: Comparar RMSE, MAE e MAPE globais no conjunto de validação
#            (2022–2023) entre os 3 modelos da Fase 5, e antecipar o MAPE no
#            teste (2024+) para ver se a hierarquia se mantém fora da amostra.
# Objectivo: Eleger o modelo vencedor para a avaliação final e o backtesting.
#
# Input:  `preds` (STEP 1)
# Output: Tabela modelo × métricas na validação + MAPE no teste

linhas = []
for modelo in ["linreg", "rf", "xgb"]:
    g_val = preds[(preds["modelo"] == modelo) & (preds["split"] == "val")]
    g_test = preds[(preds["modelo"] == modelo) & (preds["split"] == "test")]
    linhas.append({
        "Modelo": MODELOS_PT[modelo],
        "RMSE val": round(np.sqrt((g_val["erro"] ** 2).mean()), 2),
        "MAE val": round(g_val["erro"].abs().mean(), 2),
        "MAPE val (%)": round(g_val["ape"].mean(), 2),
        "MAPE test (%)": round(g_test["ape"].mean(), 2),
        "Sessões val": len(g_val),
    })

tab = pd.DataFrame(linhas)
vencedor = tab.loc[tab["MAPE val (%)"].idxmin(), "Modelo"]
print(f"Vencedor na validação (menor MAPE): {vencedor}")
display(tab)

In [0]:
# ==============================================================================
# Q2: EM QUE TICKERS A PREVISÃO É MAIS FIÁVEL?
# ==============================================================================
# Descrição: MAPE por ticker na validação para cada modelo, ordenado do melhor
#            para o pior, com o melhor modelo de cada ticker.
# Objectivo: Identificar onde as previsões são accionáveis (ETFs calmos) e
#            onde falham (cripto/NVDA) — define onde usar o modelo.
#
# Input:  `preds` (STEP 1, split = val)
# Output: Tabela ticker × MAPE por modelo + melhor modelo

val = preds[preds["split"] == "val"]

tab = val.pivot_table(index="ticker", columns="modelo_pt", values="ape", aggfunc="mean").reset_index()
tab["Melhor MAPE (%)"] = tab[["LinReg", "RF", "XGBoost"]].min(axis=1).round(2)
tab["Melhor modelo"] = tab[["LinReg", "RF", "XGBoost"]].idxmin(axis=1)
tab = tab.sort_values("Melhor MAPE (%)").reset_index(drop=True)
for c in ["LinReg", "RF", "XGBoost"]:
    tab[c] = tab[c].round(2)

print(f"Mais fiáveis: {', '.join(tab['ticker'].head(3))}")
print(f"Menos fiáveis: {', '.join(tab['ticker'].tail(3))}")
display(tab)

In [0]:
# ==============================================================================
# Q3: O MODELO ACERTA A DIRECÇÃO DO MERCADO?
# ==============================================================================
# Descrição: Percentagem de sessões em que o sinal de subida/descida previsto
#            (y_prev vs close de hoje) coincide com o movimento real de t→t+1,
#            por modelo e por split. Referência: acertar ao calhas ≈ 50%.
# Objectivo: Medir a utilidade prática para trading — mais importante que o
#            erro absoluto em dólares.
#
# Input:  `preds` (STEP 1 — colunas dir_real / dir_prev / acertou)
# Output: Tabela modelo × acerto de direcção (val e test) + acerto por ticker no test

tab = (preds.pivot_table(index="modelo_pt", columns="split", values="acertou", aggfunc="mean") * 100).round(1)
tab = tab.reset_index().rename(columns={"modelo_pt": "Modelo",
                                        "val": "Acerto val (%)",
                                        "test": "Acerto test (%)"})
print("Acerto de direcção por modelo (referência: 50% = acertar ao calhas):")
display(tab)

tab_ticker = (preds[preds["split"] == "test"]
              .pivot_table(index="ticker", columns="modelo_pt", values="acertou", aggfunc="mean") * 100).round(1)
tab_ticker = tab_ticker.reset_index().rename(columns={"ticker": "Ticker"})
print("Acerto de direcção por ticker no TESTE (2024+), por modelo:")
display(tab_ticker.sort_values("Ticker"))

In [0]:
# ==============================================================================
# Q4: AS PREVISÕES GERAM RETORNO SUPERIOR AO BUY & HOLD?
# ==============================================================================
# Descrição: Backtest simples no conjunto de teste (2024+): comprar ao fecho de
#            hoje quando o modelo prevê subida e vender ao fecho de amanhã;
#            ficar em caixa quando prevê descida. Retorno acumulado por modelo
#            vs buy & hold, por ticker (sem custos de transacção).
# Objectivo: Responder à pergunta central do negócio — "isto serve para investir?"
#
# Input:  `preds` (STEP 1 — split test, colunas dir_prev / ret_dia)
# Output: Tabela ticker: buy & hold vs estratégia por modelo + melhor estratégia

teste = preds[preds["split"] == "test"].copy()
teste["ret_estrategia"] = np.where(teste["dir_prev"] > 0, teste["ret_dia"], 0.0)

# Buy & hold por ticker (igual para os 3 modelos)
bh = teste.groupby("ticker")["ret_dia"].agg(lambda r: (1 + r).prod() - 1)

# Retorno acumulado da estratégia por ticker × modelo
est = teste.groupby(["ticker", "modelo_pt"])["ret_estrategia"].agg(lambda r: (1 + r).prod() - 1).unstack("modelo_pt")

tab = pd.concat([bh.rename("Buy & Hold"), est], axis=1).reset_index()
tab["Melhor estratégia"] = tab[["Buy & Hold", "LinReg", "RF", "XGBoost"]].idxmax(axis=1)
tab = tab.sort_values("Buy & Hold", ascending=False).reset_index(drop=True)

n_ganha = {m: int((tab[m] > tab["Buy & Hold"]).sum()) for m in ["LinReg", "RF", "XGBoost"]}
print("Tickers em que a estratégia ganha ao buy & hold:",
      f"LinReg {n_ganha['LinReg']}/12 | RF {n_ganha['RF']}/12 | XGBoost {n_ganha['XGBoost']}/12")

for c in ["Buy & Hold", "LinReg", "RF", "XGBoost"]:
    tab[c] = (tab[c] * 100).round(1)
tab.columns = ["Ticker", "Buy & Hold (%)", "Estr. LinReg (%)", "Estr. RF (%)", "Estr. XGBoost (%)", "Melhor estratégia"]
display(tab)

In [0]:
# ==============================================================================
# Q5: O MODELO FUNCIONA MELHOR EM ACÇÕES, ETFs OU CRIPTOMOEDAS?
# ==============================================================================
# Descrição: MAPE médio por classe de activo (Acções, ETF, Cripto) e por modelo,
#            na validação e no teste.
# Objectivo: Decidir se valeria a pena um modelo dedicado por classe de activo
#            e onde o modelo único multi-ticker se justifica.
#
# Input:  `preds` (STEP 1 — coluna classe)
# Output: Tabela classe × modelo (MAPE %) + sessões por classe

def tabela_classe(split):
    g = preds[preds["split"] == split]
    tab = g.pivot_table(index="classe", columns="modelo_pt", values="ape", aggfunc="mean").round(2)
    tab["Sessões (por modelo)"] = (g.groupby("classe").size() // 3).astype(int)
    return tab.reset_index().rename(columns={"classe": "Classe"})

print("MAPE (%) por classe — VALIDAÇÃO (2022–2023):")
display(tabela_classe("val"))
print("MAPE (%) por classe — TESTE (2024+):")
display(tabela_classe("test"))

In [0]:
# ==============================================================================
# Q6: O ERRO AUMENTA COM O TEMPO?
# ==============================================================================
# Descrição: MAPE por ano civil e por modelo, do validação (2022–2023) ao teste
#            (2024–2026) — o modelo foi treinado só até fim de 2021.
# Objectivo: Medir degradação/deriva do modelo e a necessidade de re-treino
#            periódico em produção.
#
# Input:  `preds` (STEP 1) + ano civil da data
# Output: Tabela ano × modelo (MAPE %) + variação val → test

preds["ano"] = pd.to_datetime(preds["date"]).dt.year

tab = (preds.pivot_table(index="ano", columns="modelo_pt", values="ape", aggfunc="mean")
           .round(2).reset_index().rename(columns={"ano": "Ano"}))

print("MAPE (%) por ano (val = 2022–2023 | test = 2024–2026, nunca visto no treino):")
display(tab)

val_m = preds[preds["split"] == "val"].groupby("modelo_pt")["ape"].mean()
test_m = preds[preds["split"] == "test"].groupby("modelo_pt")["ape"].mean()
delta = ((test_m - val_m) / val_m * 100).round(1)
print("Variação do MAPE do val para o test:",
      " | ".join(f"{m}: {v:+.1f}%" for m, v in delta.items()))

In [0]:
# ==============================================================================
# Q7: O MODELO ERRA MAIS QUANDO O MERCADO SOBE OU QUANDO DESCE?
# ==============================================================================
# Descrição: Erro médio (y_prev − y_real) e MAE por direcção real do mercado
#            (dias de subida vs descida), por modelo. Erro médio negativo em
#            dias de subida = o modelo subestima as altas.
# Objectivo: Perceber o viés do modelo (ex.: árvores não extrapolam tendências)
#            e as implicações para gestão de risco.
#
# Input:  `preds` (STEP 1 — colunas dir_real / erro, val + test)
# Output: Tabela modelo × direcção: erro médio, MAE e n.º de sessões

g = preds[preds["dir_real"] != 0]  # exclui dias exactamente planos
tab = (g.groupby(["modelo_pt", g["dir_real"].map({1: "Subida", -1: "Descida"})])
       .agg(erro_medio=("erro", "mean"), MAE=("erro", lambda e: e.abs().mean()),
            sessoes=("erro", "size")).reset_index())
tab.columns = ["Modelo", "Direcção real", "Erro médio", "MAE", "Sessões"]
tab["Erro médio"] = tab["Erro médio"].round(2)
tab["MAE"] = tab["MAE"].round(2)
tab = tab.sort_values(["Modelo", "Direcção real"]).reset_index(drop=True)
display(tab)

In [0]:
# ==============================================================================
# Q8: QUAIS AS FEATURES MAIS IMPORTANTES PARA PREVER O PREÇO?
# ==============================================================================
# Descrição: Re-treinar RF e XGBoost com a arquitectura exacta da Fase 5
#            (modelo único multi-ticker, ticker em one-hot) e comparar a
#            importância das features. Os 12 dummies de ticker são agregados
#            numa só linha "ticker (one-hot)".
# Objectivo: Explicar o que os modelos aprenderam — interpretabilidade de
#            negócio: o preço de ontem domina? O RSI/MACD ajudam?
#
# Input:  workspace.market_project.gold_features_split (train ≤ 2021)
# Output: Tabela feature × importância RF e XGBoost (%), ordenada pela RF

FEATURES_NUM = ["ret_log", "sma_5", "sma_10", "sma_20", "sma_50", "vol_20",
                "rsi_14", "macd", "macd_signal", "macd_hist", "vol_rel_20",
                "close_lag_1", "close_lag_2", "close_lag_3", "close_lag_4", "close_lag_5"]
ALVO = "target_close_next"
X_COLS = ["ticker"] + FEATURES_NUM

train_q8 = (spark.table(f"{SCHEMA}.gold_features_split")
            .select("ticker", "split", *FEATURES_NUM, ALVO).toPandas())
train_q8 = train_q8[train_q8["split"] == "train"].reset_index(drop=True)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

preproc = ColumnTransformer([
    ("cat_ticker", OneHotEncoder(handle_unknown="ignore"), ["ticker"]),
    ("num", "passthrough", FEATURES_NUM),
])

pipe_rf_q8 = Pipeline([("prep", preproc),
                       ("modelo", RandomForestRegressor(n_estimators=300, max_depth=12,
                                                         min_samples_leaf=5, random_state=42,
                                                         n_jobs=-1))])
pipe_xgb_q8 = Pipeline([("prep", preproc),
                        ("modelo", XGBRegressor(n_estimators=500, learning_rate=0.05,
                                                max_depth=6, subsample=0.8,
                                                colsample_bytree=0.8, random_state=42))])

pipe_rf_q8.fit(train_q8[X_COLS], train_q8[ALVO])
pipe_xgb_q8.fit(train_q8[X_COLS], train_q8[ALVO])

nomes = list(pipe_rf_q8.named_steps["prep"].get_feature_names_out())

imp = pd.DataFrame({
    "Feature": ["ticker (one-hot)" if n.startswith("cat_ticker") else n.split("__")[-1] for n in nomes],
    "RF": pipe_rf_q8.named_steps["modelo"].feature_importances_,
    "XGBoost": pipe_xgb_q8.named_steps["modelo"].feature_importances_,
}).groupby("Feature", as_index=False).sum()

imp["RF"] = (imp["RF"] * 100).round(1)
imp["XGBoost"] = (imp["XGBoost"] * 100).round(1)
imp = imp.sort_values("RF", ascending=False).reset_index(drop=True)

print("Top 5 RF:", ", ".join(imp.head(5)["Feature"]))
print("Top 5 XGBoost:", ", ".join(imp.sort_values("XGBoost", ascending=False).head(5)["Feature"]))
display(imp)

In [0]:
# ==============================================================================
# Q9: QUAL O PIOR CENÁRIO DE ERRO POR TICKER?
# ==============================================================================
# Descrição: Além do MAPE médio, o erro nos 5% piores dias por ticker (P95 do
#            erro % absoluto) e o pior dia registado no teste (2024+).
# Objectivo: Gestão de risco — quantificar a cauda do erro e saber onde o
#            modelo pode falhar de forma catastrófica.
#
# Input:  `preds` (STEP 1 — split test)
# Output: Tabela ticker: P95 por modelo + pior dia (data, modelo e erro %)

teste = preds[preds["split"] == "test"]

p95 = teste.pivot_table(index="ticker", columns="modelo_pt", values="ape",
                        aggfunc=lambda s: s.quantile(0.95)).round(2).reset_index()

pior_idx = teste.groupby("ticker")["ape"].idxmax()
pior = teste.loc[pior_idx, ["ticker", "date", "modelo_pt", "ape"]].rename(
    columns={"date": "pior_dia", "modelo_pt": "pior_modelo", "ape": "pior_ape"})

tab = p95.merge(pior, on="ticker").sort_values("RF", ascending=False).reset_index(drop=True)
tab["pior_dia"] = pd.to_datetime(tab["pior_dia"]).dt.strftime("%Y-%m-%d")
tab["pior_ape"] = tab["pior_ape"].round(2)
tab.columns = ["Ticker", "P95 LinReg (%)", "P95 RF (%)", "P95 XGBoost (%)",
               "Pior dia", "Pior modelo", "Erro pior dia (%)"]
display(tab)

# Conclusões de Negócio (3 modelos — LSTM pendente)

1. **Melhor modelo (Q1/Q6):** o RF ganha na validação (MAPE 3,66%) mas degrada-se no teste (9,30%; +153,8% vs val) — árvores não extrapolam tendências. O LinReg é o mais estável: MAPE desce de 4,93% (val) para 2,70% (test). XGBoost: 3,94% → 8,10%.
2. **Fiabilidade por ticker (Q2):** ETFs calmos no topo (IAU 1,14%, SPY 1,52%, QQQ 1,54% na val); NVDA/TSLA/META e cripto no fundo. BTC e ETH: só o LinReg é competitivo (1,92% / 2,57%).
3. **Direcção (Q3):** acerto de 48,7%–51,7% — praticamente o nível do acaso (~50%). Os modelos prevêem o *nível* do preço, não o *sinal* de trading.
4. **Backtest (Q4):** a estratégia "comprar quando prevê subida" perde contra o buy & hold em 11/12 tickers; só o RF no ETH ganha (51,3% vs −13,5%). O fecho de amanhã aproxima-se de um random walk.
5. **Classes (Q5):** na val, o RF domina em acções (2,54%) e ETFs (1,40%) e o LinReg na cripto (2,25% vs 8,69% do RF). No teste, o LinReg passa à frente em todas as classes (acções 2,66%, cripto 2,11%, ETF 3,38%).
6. **Viés (Q7):** RF/XGBoost subestimam sistematicamente (erro médio fortemente negativo, sobretudo em dias de subida: −1.233/−1.444 USD); o LinReg é simétrico (±125 USD).
7. **Features (Q8):** as médias móveis e o `close_lag_1` dominam; RSI, MACD, volume e ticker têm importância ≈ 0 — o preço de ontem é o que mais explica o de amanhã.
8. **Risco (Q9):** na cauda, o RF falha até 61–74% num único dia em SPY/QQQ/META (preços 2024+ fora do intervalo de treino ≤2021); o LinReg tem cauda mais curta, excepto IAU (87% num dia — preço baixo amplifica o erro %).

**Resposta de negócio:** com estes 3 modelos, as previsões servem para estimar o nível do preço em activos calmos (ETFs/acções, LinReg), não para gerar sinais de trading superiores ao buy & hold. Pendente: LSTM (Fase 5 4/4) e Fase 6 (avaliação final com custos de transacção).